# GloVe-Twitter + BiLSTM sentiment classification
**Contributor: Yuchen Gu | DSAI4205 | Four-class text classification**

This notebook contains the complete implementation of my BiLSTM pipeline, with English explanations for integration into the group submission. It includes preprocessing **after the shared cleaning stage**, vocabulary construction, embedding initialization, model selection, refitting, optional continuation, evaluation, error analysis, and recorded iterations. It does not implement a separate mean-pooled MLP model.

**Main configuration:** pretrained GloVe-Twitter-50; a one-layer bidirectional LSTM with 48 hidden units per direction; masked mean pooling; Linear(96,96) → ReLU → Dropout(0.4) → Linear(96,4). Embeddings are fine-tuned from epoch 1. Cross-entropy receives logits; softmax is used only to display probabilities.

**Run:** install dependencies, select a Python kernel, then use **Restart Kernel and Run All**. Use the local cleaned CSVs and shared split manifest as described in [the model README](../../models/yuchen/README.md). To inspect saved results, open the notebook without running cells. GloVe is loaded from the Gensim cache or downloaded on first use. CUDA is used when available, with a CPU fallback. Each run writes to a new `bilstm_outputs/` directory. Saved image outputs are cleared; execution generates all metrics and figures.



## Environment and configuration
Install missing libraries in the notebook's kernel environment. A working CUDA-enabled PyTorch installation should be retained. If PyTorch is absent, install the appropriate build using [the PyTorch installation guide](https://pytorch.org/get-started/locally/).

The default configuration reproduces the 50-dimensional experiment protocol. `EXTRA_EPOCHS = 20` performs a further 20 epochs on the refit model, without resetting its weights or optimizer. Leave it at zero for the validation-selected training duration. Existing output directories are never overwritten.

In [1]:
# Uncomment only if the required packages are missing.
# %pip install numpy pandas scikit-learn matplotlib nbformat ipykernel
# %pip install gensim  # Required to load/download GloVe vectors.
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
import csv, hashlib, json, random, re, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from IPython.display import display, Image, Markdown

CONFIG = dict(embedding="glove-twitter-50", dimension=50, hidden_size=48,
              dropout=0.4, batch_size=32, seed=42, epochs=40, patience=8,
              freeze_epochs=0, embedding_lr=5e-5, network_lr=8e-4, weight_decay=1e-4)
EXTRA_EPOCHS = 0
assert CONFIG["dimension"] in (25, 50, 100, 200)
assert CONFIG["embedding"] == f"glove-twitter-{CONFIG['dimension']}"
assert isinstance(EXTRA_EPOCHS, int) and EXTRA_EPOCHS >= 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_IDS = [0, 1, 2, 3]
LABEL_NAMES = ["Negative", "Positive", "Irrelevant", "Neutral"]
TOKEN_RE = re.compile(r"[A-Za-z0-9_']+")
PAD, UNK = "<pad>", "<unk>"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print({"device": str(device), "torch": str(torch.__version__),
       "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name() if device.type == "cuda" else None})
display(pd.Series(CONFIG, name="value").to_frame())

{'device': 'cuda', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 4080 Laptop GPU'}


,value
embedding,glove-twitter-50
dimension,50
hidden_size,48
dropout,0.4
batch_size,32
seed,42
epochs,40
patience,8
freeze_epochs,0
embedding_lr,0.00005


In [2]:
# Expected hashes of the columns used in the original run, in their original order.
DATA_SHA256 = {
    "train": "174adaf78e0225329cc828b5441e4976555e05df9399f396e6f1f05a261438d7",
    "test": "8797d197fc57919748c6efe594bf4dc94dd048ef56dc58a096e8cbc292ba7f47",
    "split": "e7fc1c5525dc8d9316e755c67b47dbe574115ef68959f0f33ece2e732f1f5bfb"
}

In [3]:
def data_directory():
    override = os.environ.get("DSAI4205_DATA_DIR")
    if override:
        return Path(override).expanduser()
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for directory in (base / "data/processed", base / "group project/data/processed"):
            if (directory / "train_cleaned.csv").is_file():
                return directory
    raise FileNotFoundError("Set DSAI4205_DATA_DIR to the directory with the cleaned CSVs and split_manifest.csv.")


def read_local_data(name):
    if name == "split":
        filename = "split_manifest.csv"
        columns = ["row_id", "Sentiment", "group_id", "partition"]
    else:
        filename = f"{name}_cleaned.csv"
        columns = ["row_id", "Sentiment", "text_clean", "Phrase"]
    frame = pd.read_csv(data_directory() / filename, keep_default_na=False).loc[:, columns]
    serialized = frame.to_csv(index=False, lineterminator="\n").encode("utf-8")
    if hashlib.sha256(serialized).hexdigest() != DATA_SHA256[name]:
        raise ValueError(f"Data differs from the archived run: {filename}")
    return frame


RUN_DIR = Path("bilstm_outputs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Output folder:", RUN_DIR.name)

Output folder: 20261121T165235_181853Z


## Data contract and tokenization
The shared preprocessing pipeline supplies `text_clean`, labels, and stable `row_id` values. This notebook uses the supplied text unchanged, lowercases it during tokenization, and extracts tokens with `[A-Za-z0-9_']+`. GloVe does not supply a mandatory tokenizer. No additional stemming or stop-word removal is introduced here.

The fixed training pool has **591 rows**, split into **473 fitting rows and 118 validation rows**. Early stopping uses validation macro-F1 only. During selection, the vocabulary is built from the 473 fitting texts. After the epoch count is selected, a fresh vocabulary and model are fitted on all 591 rows. Test text never contributes to either vocabulary.

All primary evaluation results use the **397-row test file**. This set has already been inspected repeatedly during development, including training-length comparisons; it is therefore not a fresh, untouched final holdout. This notebook does not claim a 372-row filtered evaluation or construct an alternative overlap filter. A group-wide filtered comparison requires the shared overlap manifest and identical row IDs across models.

Unknown words map to ID 1; padding uses ID 0. Empty text maps to one UNK token, and sequences are padded only to the longest sequence in the batch. There is no length truncation.

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def pool_fingerprint(pool):
    data = pool[["row_id", "text_clean", "Sentiment"]].to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def load_data():
    pool = read_local_data("train")
    test = read_local_data("test")
    manifest = read_local_data("split").set_index("row_id")
    if set(pool.row_id) != set(manifest.index):
        raise ValueError("Training pool and split manifest row IDs do not match")
    train = pool[pool.row_id.map(manifest.partition) == "train"].reset_index(drop=True)
    valid = pool[pool.row_id.map(manifest.partition) == "validation"].reset_index(drop=True)
    if len(train) != 473 or len(valid) != 118 or len(test) != 397:
        raise ValueError(f"Unexpected fixed split sizes: train={len(train)}, validation={len(valid)}, test={len(test)}")
    return pool.reset_index(drop=True), train, valid, test


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(str(text).lower())


def build_vocab(texts) -> dict[str, int]:
    vocab = {PAD: 0, UNK: 1}
    for text in texts:
        for token in tokenize(text):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab


class TweetDataset(Dataset):
    def __init__(self, frame, vocab):
        self.texts = frame.text_clean.astype(str).tolist()
        self.labels = frame.Sentiment.astype(int).tolist()
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        ids = [self.vocab.get(t, self.vocab[UNK]) for t in tokenize(self.texts[index])]
        if not ids:
            ids = [self.vocab[UNK]]
        return torch.tensor(ids, dtype=torch.long), self.labels[index]


def collate(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)
    padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)


def make_loader(frame, vocab, batch_size, shuffle):
    return DataLoader(TweetDataset(frame, vocab), batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

In [5]:
pool, training, validation, test = load_data()
for frame in (pool, training, validation, test):
    assert frame.row_id.is_unique
    assert frame.Sentiment.isin(LABEL_IDS).all()
assert set(training.row_id).isdisjoint(validation.row_id)
assert set(pool.row_id).isdisjoint(test.row_id)
split_table = pd.DataFrame({name: frame.Sentiment.value_counts().reindex(LABEL_IDS, fill_value=0)
    for name, frame in [("train", training), ("validation", validation), ("refit_pool", pool), ("test", test)]})
split_table.index = LABEL_NAMES
display(split_table)
display(training[["row_id", "Sentiment"]].head(3))
for name, frame in [("training", training), ("validation", validation), ("test", test)]:
    frame[["row_id", "Sentiment"]].to_csv(RUN_DIR / f"{name}_ids.csv", index=False)


,train,validation,refit_pool,test
Negative,138,36,174,120
Positive,134,31,165,106
Irrelevant,80,26,106,66
Neutral,121,25,146,105


,row_id,Sentiment
0,train_0,0
1,train_1,0
2,train_2,0


## GloVe word representations
GloVe learns word vectors from global co-occurrence statistics. Here externally pretrained Twitter GloVe initializes the task embedding table, which is fine-tuned using sentiment loss. This is not training GloVe from scratch.

The first use downloads the selected archive; subsequent runs use the local Gensim cache. Only words in the fitting partition enter the task vocabulary. Matched words use pretrained vectors; unmatched words and UNK use seeded Gaussian initialization (standard deviation 0.08), and PAD is zero. Coverage is reported below. No word vectors or trained checkpoint are embedded in this notebook.

Lookup converts padded token IDs into word-vector sequences. To try another embedding, update both `dimension` and `embedding` in the configuration.

In [6]:
_external_vectors = None

def initialize_embeddings(frame, phase):
    """Use only the fitting partition to define IDs, then initialize those IDs."""
    global _external_vectors
    vocab = build_vocab(frame.text_clean)
    if _external_vectors is None:
        import gensim.downloader as api
        from gensim.models import KeyedVectors
        local = Path(api.BASE_DIR) / CONFIG["embedding"] / (CONFIG["embedding"] + ".gz")
        _external_vectors = (KeyedVectors.load_word2vec_format(str(local))
                             if local.exists() else api.load(CONFIG["embedding"]))
    matrix = np.random.default_rng(CONFIG["seed"]).normal(
        0, 0.08, (len(vocab), CONFIG["dimension"])).astype(np.float32)
    matrix[0] = 0
    matched = 0
    for token, idx in vocab.items():
        if idx > 1 and token in _external_vectors:
            matrix[idx] = _external_vectors[token]
            matched += 1
    metadata = dict(source=CONFIG["embedding"], dimension=CONFIG["dimension"],
                    task_vocab=len(vocab), matched_words=matched,
                    coverage=matched / max(1, len(vocab)-2))
    assert matrix.shape == (len(vocab), CONFIG["dimension"])
    assert np.isfinite(matrix).all() and np.all(matrix[0] == 0)
    dest = RUN_DIR / phase
    dest.mkdir(exist_ok=True)
    np.save(dest / "embeddings.npy", matrix)
    (dest / "vocab.json").write_text(json.dumps(vocab), encoding="utf-8")
    (dest / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    display(pd.Series(metadata, name=phase).to_frame())
    return vocab, matrix

selection_vocab, selection_matrix = initialize_embeddings(training, "selection_glove")
assert selection_vocab == build_vocab(training.text_clean)

,selection_glove
source,glove-twitter-50
dimension,50
task_vocab,2356
matched_words,2140
coverage,0.909091


## Independent BiLSTM classifier
**Text → token IDs → GloVe lookup → BiLSTM → masked mean → dense/ReLU/dropout → four logits.**

The BiLSTM reads the sequence in both directions. At each token, its forward and backward hidden states are concatenated into a 96-dimensional vector. Packed sequences prevent padding from participating in recurrence. Masked mean pooling produces one 96-dimensional representation per text. A hidden dense layer, ReLU and dropout precede the four-class output layer. The embedding layer and the independently defined classifier are separate components.

Dropout and AdamW weight decay provide regularization. Gradient clipping limits the gradient norm to 1.0 for optimization stability. Class-weighted cross-entropy gives less frequent classes more weight. Weights are recomputed using the fitting partition in each stage.

In [7]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, matrix: np.ndarray, hidden: int, dropout: float, classes: int = 4):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(matrix), freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(matrix.shape[1], hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.hidden = nn.Linear(hidden * 2, hidden * 2)
        self.output = nn.Linear(hidden * 2, classes)

    def forward(self, ids, lengths):
        embedded = self.embedding(ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=ids.shape[1])
        mask = (ids != 0).unsqueeze(-1)
        pooled = (outputs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        features = self.dropout(torch.relu(self.hidden(pooled)))
        return self.output(features)

In [8]:
def setup(frame, matrix):
    set_seed(CONFIG["seed"])
    model = BiLSTMClassifier(matrix, CONFIG["hidden_size"], CONFIG["dropout"]).to(device)
    optimizer = torch.optim.AdamW([
        {"params": model.embedding.parameters(), "lr": CONFIG["embedding_lr"]},
        {"params": [p for n, p in model.named_parameters() if n != "embedding.weight"],
         "lr": CONFIG["network_lr"]},
    ], weight_decay=CONFIG["weight_decay"])
    weights = compute_class_weight("balanced", classes=np.array(LABEL_IDS), y=frame.Sentiment)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))
    return model, optimizer, loss_fn

# Verify padding invariance and output dimensions before running the experiment.
probe, _, _ = setup(training, selection_matrix)
probe.eval()
with torch.no_grad():
    a = probe(torch.tensor([[1, 2]], device=device), torch.tensor([2]))
    b = probe(torch.tensor([[1, 2, 0, 0]], device=device), torch.tensor([2]))
    torch.testing.assert_close(a, b, atol=1e-6, rtol=1e-5)
    assert a.shape == (1, 4)
del probe
print("Verified padding invariance and four-logit output.")

Verified padding invariance and four-logit output.


## Training, logging, and checkpoint helpers
The same training functions are used for selection, refitting and optional continuation. Every epoch logs loss, accuracy and macro-F1. Training metrics are aggregated from batches with dropout active; validation/test metrics use the fixed model in evaluation mode.

The maximum selection budget is 40 epochs, with patience 8 and seed 42. The best epoch is chosen by validation macro-F1. Test diagnostics are recorded only during refitting/continuation and do not drive stopping or selection in this code. Logging test metrics preserves training RNG state. A checkpoint saves model weights, optimizer state, vocabulary, configuration, row fingerprint, and RNG state.

In [9]:
def evaluate(model, loader, device, loss_fn=None):
    model.eval()
    ys, preds = [], []
    loss_sum, loss_weight = 0.0, 0.0
    with torch.no_grad():
        for ids, lengths, labels in loader:
            logits = model(ids.to(device), lengths)
            if loss_fn is not None:
                targets = labels.to(device)
                weight = loss_fn.weight[targets].sum().item() if loss_fn.weight is not None else len(labels)
                loss_sum += loss_fn(logits, targets).item() * weight
                loss_weight += weight
            preds.extend(logits.argmax(1).cpu().tolist())
            ys.extend(labels.tolist())
    if loss_fn is None:
        return np.asarray(ys), np.asarray(preds)
    return np.asarray(ys), np.asarray(preds), loss_sum / loss_weight


def test_snapshot(model, loader, device, loss_fn):
    """Observe test metrics without advancing training RNG or changing model mode."""
    was_training = model.training
    devices = list(range(torch.cuda.device_count())) if device.type == "cuda" else []
    try:
        # Even a non-shuffled DataLoader consumes a random base seed on iteration.
        with torch.random.fork_rng(devices=devices):
            y, pred, loss = evaluate(model, loader, device, loss_fn)
    finally:
        model.train(was_training)
    return {
        "test_loss": round(float(loss), 5),
        "test_accuracy": round(float(accuracy_score(y, pred)), 4),
        "test_macro_f1": round(float(f1_score(y, pred, labels=LABEL_IDS, average="macro", zero_division=0)), 4),
        "test_weighted_f1": round(float(f1_score(y, pred, labels=LABEL_IDS, average="weighted", zero_division=0)), 4),
    }


def train_epoch(model, loader, optimizer, loss_fn, device):
    """Metrics use the same training-mode predictions used for each update."""
    model.train()
    loss_sum, loss_weight = 0.0, 0.0
    ys, preds = [], []
    for ids, lengths, labels in loader:
        ids, labels = ids.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(ids, lengths)
        loss = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        # CrossEntropyLoss(mean) normalizes by target class weights, not batch size.
        weight = loss_fn.weight[labels].sum().item() if loss_fn.weight is not None else len(labels)
        loss_sum += loss.item() * weight
        loss_weight += weight
        ys.extend(labels.detach().cpu().tolist())
        preds.extend(logits.detach().argmax(1).cpu().tolist())
    return {
        "train_loss": round(loss_sum / loss_weight, 5),
        "train_accuracy": round(float(accuracy_score(ys, preds)), 4),
        "train_macro_f1": round(float(f1_score(ys, preds, labels=LABEL_IDS, average="macro", zero_division=0)), 4),
    }

In [10]:
class EpochLogger:
    """Close files after each epoch so completed records survive interruptions."""
    fields = [
        "phase", "epoch", "recorded_at_utc", "train_loss", "train_accuracy", "train_macro_f1",
        "validation_loss", "validation_accuracy", "validation_macro_f1", "epoch_seconds",
        "embedding_trainable", "embedding_lr", "network_lr",
        "test_loss", "test_accuracy", "test_macro_f1", "test_weighted_f1",
    ]

    def __init__(self, directory):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)

    def record(self, phase, epoch, metrics, model, optimizer, started):
        row = dict.fromkeys(self.fields)
        row.update(metrics)
        row.update({
            "phase": phase, "epoch": epoch,
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
            "epoch_seconds": round(time.perf_counter() - started, 3),
            "embedding_trainable": model.embedding.weight.requires_grad,
            "embedding_lr": optimizer.param_groups[0]["lr"],
            "network_lr": optimizer.param_groups[1]["lr"],
        })
        csv_path = self.directory / "epochs.csv"
        needs_header = not csv_path.exists() or csv_path.stat().st_size == 0
        with csv_path.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=self.fields)
            if needs_header:
                writer.writeheader()
            writer.writerow(row)
        with (self.directory / "epochs.jsonl").open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
        message = (f"{phase} epoch {epoch:02d}: loss={row['train_loss']:.5f}, "
                   f"accuracy={row['train_accuracy']:.4f}, macro_f1={row['train_macro_f1']:.4f}")
        if row["validation_loss"] is not None:
            message += (f", validation_loss={row['validation_loss']:.5f}, "
                        f"validation_macro_f1={row['validation_macro_f1']:.4f}")
        if row["test_accuracy"] is not None:
            message += f", test_accuracy={row['test_accuracy']:.4f}, test_macro_f1={row['test_macro_f1']:.4f}"
        print(message + f", seconds={row['epoch_seconds']:.3f}", flush=True)
        return row


def run_epochs(model, train_loader, valid_loader, optimizer, loss_fn, device, epochs, patience, freeze_epochs, logger):
    best_score, best_epoch, best_state, stale = -1.0, 0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        model.embedding.weight.requires_grad_(epoch > freeze_epochs)
        metrics = train_epoch(model, train_loader, optimizer, loss_fn, device)
        y, pred, valid_loss = evaluate(model, valid_loader, device, loss_fn)
        score = float(f1_score(y, pred, labels=LABEL_IDS, average="macro", zero_division=0))
        metrics.update({"validation_loss": round(valid_loss, 5),
                        "validation_accuracy": round(float(accuracy_score(y, pred)), 4),
                        "validation_macro_f1": round(score, 4)})
        history.append(logger.record("selection", epoch, metrics, model, optimizer, started))
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)
    return best_epoch, best_score, history


def save_refit_checkpoint(path, model, optimizer, vocab, config, epoch, history, pool):
    """Save at an epoch boundary, before evaluation consumes random numbers."""
    numpy_state = np.random.get_state()
    checkpoint = {
        "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "optimizer_state_dict": optimizer.state_dict(),
        "vocab": vocab, "config": dict(config),
        "embedding_dimension": model.embedding.embedding_dim,
        "phase": "refit", "completed_epochs": epoch, "refit_history": history,
        "pool_fingerprint": pool_fingerprint(pool),
        "rng_state": {
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "python": random.getstate(),
            "numpy": [numpy_state[0], numpy_state[1].tolist(), numpy_state[2], numpy_state[3], numpy_state[4]],
        },
    }
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def restore_rng_state(state):
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([x.cpu() for x in state["cuda"]])
    random.setstate(state["python"])
    kind, keys, pos, has_gauss, cached = state["numpy"]
    np.random.set_state((kind, np.asarray(keys, dtype=np.uint32), pos, has_gauss, cached))

## Select the training duration on validation data
No test result is used in this cell. A fresh model is fitted on 473 rows; the selected weights and validation report are retained for inspection.

In [11]:
runtime = dict(config=CONFIG, device=str(device), torch_version=str(torch.__version__),
               gpu=torch.cuda.get_device_name() if device.type == "cuda" else None,
               data_sha256=DATA_SHA256,
               test_role="397 previously inspected rows; diagnostic, not a fresh holdout",
               extra_epochs=EXTRA_EPOCHS)
(RUN_DIR / "config.json").write_text(json.dumps(runtime, indent=2), encoding="utf-8")
model, optimizer, loss_fn = setup(training, selection_matrix)
logger = EpochLogger(RUN_DIR)
selected_epoch, best_validation_f1, selection_history = run_epochs(
    model, make_loader(training, selection_vocab, CONFIG["batch_size"], True),
    make_loader(validation, selection_vocab, CONFIG["batch_size"], False),
    optimizer, loss_fn, device, CONFIG["epochs"], CONFIG["patience"], CONFIG["freeze_epochs"], logger)
torch.save({"state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "vocab": selection_vocab, "selected_epoch": selected_epoch, "config": CONFIG},
           RUN_DIR / "selection_best.pt")
pd.DataFrame(selection_history).to_csv(RUN_DIR / "selection_history.csv", index=False)
print(f"Selected epoch: {selected_epoch}; validation macro-F1: {best_validation_f1:.4f}")

selection epoch 01: loss=1.38927, accuracy=0.2368, macro_f1=0.1754, validation_loss=1.38316, validation_macro_f1=0.1724, seconds=0.365


selection epoch 02: loss=1.37968, accuracy=0.3150, macro_f1=0.3084, validation_loss=1.37933, validation_macro_f1=0.2732, seconds=0.294


selection epoch 03: loss=1.37708, accuracy=0.3615, macro_f1=0.3429, validation_loss=1.37426, validation_macro_f1=0.3737, seconds=0.295


selection epoch 04: loss=1.37081, accuracy=0.3932, macro_f1=0.3381, validation_loss=1.36550, validation_macro_f1=0.3588, seconds=0.272


selection epoch 05: loss=1.35428, accuracy=0.4355, macro_f1=0.3706, validation_loss=1.35561, validation_macro_f1=0.3600, seconds=0.218


selection epoch 06: loss=1.32969, accuracy=0.4884, macro_f1=0.4587, validation_loss=1.32877, validation_macro_f1=0.4213, seconds=0.213


... (11 outputs left out)

selection epoch 18: loss=0.92686, accuracy=0.6490, macro_f1=0.6339, validation_loss=1.27262, validation_macro_f1=0.4499, seconds=0.202


selection epoch 19: loss=0.90521, accuracy=0.6681, macro_f1=0.6502, validation_loss=1.28677, validation_macro_f1=0.4368, seconds=0.206


selection epoch 20: loss=0.87215, accuracy=0.6956, macro_f1=0.6797, validation_loss=1.35538, validation_macro_f1=0.4438, seconds=0.220


Selected epoch: 12; validation macro-F1: 0.4962


## Evaluation exports
All evaluations below use fixed class order: Negative, Positive, Irrelevant, Neutral. `P`, `R`, and `F1` in summary tables are **macro averages**, giving equal weight to each class. Per-class scores include support. The confusion matrix has actual classes on rows and predicted classes on columns. TP/FP/FN/TN are computed one class versus all others.

In [12]:
def plot_confusion_report(path, title, matrix, counts):
    names = counts["label"].tolist()
    values = np.asarray(matrix)
    total, correct = int(values.sum()), int(values.trace())
    fig = Figure(figsize=(10.6, 4.9), facecolor="white")
    fig.subplots_adjust(left=.095, right=.98, bottom=.16, top=.75, wspace=.22)
    ax, detail = fig.subplots(1, 2, gridspec_kw={"width_ratios": [1, 1.25]})
    fig.text(.065, .91, title, fontsize=17, weight="bold", color="#182e40")
    fig.text(.065, .845, f"{total:,} examples   |   {correct:,} correct   |   Accuracy {correct / total:.2%}",
             fontsize=11, color="#465969")
    heat = ax.imshow(values, cmap="Blues", vmin=0, vmax=max(1, values.max()))
    ax.set_xticks(range(len(names)), names, fontsize=9)
    ax.set_yticks(range(len(names)), names, fontsize=9)
    ax.set(xlabel="Predicted class", ylabel="Actual class")
    ax.tick_params(length=0, pad=5)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for (row, col), count in np.ndenumerate(values):
        ax.text(col, row, str(count), ha="center", va="center", fontsize=13,
                color="white" if heat.norm(count) > .62 else "#182e40",
                weight="bold" if row == col else "normal")
    detail.axis("off")
    detail.text(0, .94, "Per-class counts", fontsize=13, weight="bold", color="#182e40")
    cols = ["label", "TP", "FP", "FN", "TN", "support"]
    table = detail.table(cellText=counts[cols].values, colLabels=["Class", *cols[1:-1], "Actual n"],
                         cellLoc="center", colWidths=[.25, .14, .14, .14, .14, .19],
                         bbox=[0, .29, 1, .55])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("white")
        cell.set_facecolor("#254b64" if r == 0 else ("#eaf2f6" if r % 2 else "#f6f8fa"))
        cell.set_text_props(color="white" if r == 0 else "#182e40",
                            weight="bold" if r == 0 else "normal")
    detail.text(0, .15, "Each row treats that class as positive.", fontsize=10, color="#465969")
    fig.savefig(path, dpi=180)


def write_confusion_tables(directory, prefix, actual, predicted, title=None):
    names = ["Negative", "Positive", "Irrelevant", "Neutral"]
    labels = list(range(len(names)))
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    if actual.ndim != 1 or actual.shape != predicted.shape or actual.size == 0:
        raise ValueError("Expected one prediction per example in a nonempty split.")
    if not (np.isin(actual, labels).all() and np.isin(predicted, labels).all()):
        raise ValueError("Sentiment IDs must be in 0..3.")
    matrix = confusion_matrix(actual, predicted, labels=labels)
    tp = matrix.diagonal()
    support = matrix.sum(axis=1)
    fn, fp = support - tp, matrix.sum(axis=0) - tp
    counts = pd.DataFrame(dict(label_id=labels, label=names, TP=tp, FP=fp, FN=fn,
                               TN=actual.size - tp - fp - fn, support=support))
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(matrix, index=names, columns=names).to_csv(
        directory / f"{prefix}_confusion_matrix.csv", index_label="actual / predicted")
    counts.to_csv(directory / f"{prefix}_class_counts.csv", index=False)
    plot_confusion_report(directory / f"{prefix}_confusion_matrix.png",
                          title or prefix.replace("_", " "), matrix, counts)

In [13]:
def export_evaluation(prefix, frame, model, vocab):
    """Compute every metric from fresh predictions, display it, and export it."""
    y, pred = evaluate(model, make_loader(frame, vocab, CONFIG["batch_size"], False), device)
    report = classification_report(y, pred, labels=LABEL_IDS, target_names=LABEL_NAMES,
                                   output_dict=True, zero_division=0)
    text = classification_report(y, pred, labels=LABEL_IDS, target_names=LABEL_NAMES,
                                 digits=4, zero_division=0)
    (RUN_DIR / f"{prefix}_metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    (RUN_DIR / f"{prefix}_classification_report.txt").write_text(text, encoding="utf-8")
    frame[["row_id", "Sentiment"]].assign(
        prediction=pred, correct=y == pred).to_csv(RUN_DIR / f"{prefix}_predictions.csv", index=False)
    write_confusion_tables(RUN_DIR, prefix, y, pred, f"GloVe + BiLSTM | {prefix}")
    cm = pd.read_csv(RUN_DIR / f"{prefix}_confusion_matrix.csv", index_col=0).to_numpy()
    assert cm.sum() == len(frame) and cm.trace() == (y == pred).sum()
    summary = pd.DataFrame([
        {"average": average, "accuracy": report["accuracy"],
         "precision": report[f"{average} avg"]["precision"],
         "recall": report[f"{average} avg"]["recall"],
         "f1": report[f"{average} avg"]["f1-score"], "n": len(y)}
        for average in ("macro", "weighted")])
    per_class = pd.DataFrame({name: report[name] for name in LABEL_NAMES}).T
    per_class["support"] = per_class["support"].astype(int)
    summary.to_csv(RUN_DIR / f"{prefix}_summary.csv", index=False)
    per_class.to_csv(RUN_DIR / f"{prefix}_per_class.csv", index_label="class")
    display(Markdown(f"### {prefix}: evaluation on {len(y)} examples"))
    display(Markdown("**Overall metrics** — P, R and F1 are shown for both macro and weighted averaging."))
    display(summary.round(4))
    display(Markdown("**Per-class precision, recall, F1 and support**"))
    display(per_class.round(4))
    display(Markdown("**Confusion matrix** — rows: actual; columns: predicted."))
    display(Image(filename=str(RUN_DIR / f"{prefix}_confusion_matrix.png")))
    display(pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES))
    display(Markdown("**One-vs-rest counts for each class**"))
    display(pd.read_csv(RUN_DIR / f"{prefix}_class_counts.csv"))
    return report

validation_report = export_evaluation("validation", validation, model, selection_vocab)

### validation: evaluation on 118 examples

**Overall metrics** — P, R and F1 are shown for both macro and weighted averaging.

,average,accuracy,precision,recall,f1,n
0,macro,0.5169,0.5056,0.5151,0.4962,118
1,weighted,0.5169,0.5110,0.5169,0.5007,118


**Per-class precision, recall, F1 and support**

,precision,recall,f1-score,support
Negative,0.5667,0.4722,0.5152,36
Positive,0.5122,0.6774,0.5833,31
Irrelevant,0.4286,0.2308,0.3000,26
Neutral,0.5152,0.6800,0.5862,25


**Confusion matrix** — rows: actual; columns: predicted.

,Negative,Positive,Irrelevant,Neutral
Negative,17,10,5,4
Positive,2,21,3,5
Irrelevant,8,5,6,7
Neutral,3,5,0,17


**One-vs-rest counts for each class**

,label_id,label,TP,FP,FN,TN,support
0,0,Negative,17,13,19,69,36
1,1,Positive,21,20,10,67,31
2,2,Irrelevant,6,8,20,84,26
3,3,Neutral,17,16,8,77,25


## Refit on the 591-row training pool
After selection, both the classifier and embedding table are reinitialized from the original initialization. This stage uses all 591 training-pool rows for exactly the selected number of epochs. Validation rows now belong to the fitting pool, so their earlier score should not be treated as validation of this refitted model.

With the default settings, embeddings are trainable from the first epoch. The checkpoint at the selected training duration is preserved separately from an optional longer continuation.

In [14]:
del model, optimizer, loss_fn
vocab, matrix = initialize_embeddings(pool, "refit_glove")
_external_vectors = None  # Release the full external archive if one was loaded.
model, optimizer, loss_fn = setup(pool, matrix)
pool_loader = make_loader(pool, vocab, CONFIG["batch_size"], True)
test_loader = make_loader(test, vocab, CONFIG["batch_size"], False)
refit_history = []
for epoch in range(1, selected_epoch + 1):
    started = time.perf_counter()
    model.embedding.weight.requires_grad_(epoch > CONFIG["freeze_epochs"])
    metrics = train_epoch(model, pool_loader, optimizer, loss_fn, device)
    metrics.update(test_snapshot(model, test_loader, device, loss_fn))
    refit_history.append(logger.record("refit", epoch, metrics, model, optimizer, started))
    save_refit_checkpoint(RUN_DIR / "model.pt", model, optimizer, vocab, CONFIG,
                          epoch, refit_history, pool)
save_refit_checkpoint(RUN_DIR / "selected_duration_model.pt", model, optimizer, vocab,
                      CONFIG, selected_epoch, refit_history, pool)
assert not np.array_equal(model.embedding.weight.detach().cpu().numpy()[1:], matrix[1:])
parameters = dict(embedding=model.embedding.weight.numel(),
                  classifier=sum(p.numel() for n, p in model.named_parameters() if n != "embedding.weight"),
                  trainable=sum(p.numel() for p in model.parameters() if p.requires_grad))
display(pd.Series(parameters, name="parameters").to_frame())
selected_test_report = export_evaluation("test_selected_duration", test, model, vocab)

,refit_glove
source,glove-twitter-50
dimension,50
task_vocab,2701
matched_words,2449
coverage,0.907373


refit epoch 01: loss=1.38615, accuracy=0.2843, macro_f1=0.2192, test_accuracy=0.4181, test_macro_f1=0.3251, seconds=0.377


refit epoch 02: loss=1.37884, accuracy=0.3130, macro_f1=0.2867, test_accuracy=0.4332, test_macro_f1=0.3755, seconds=0.377


refit epoch 03: loss=1.36876, accuracy=0.3587, macro_f1=0.3267, test_accuracy=0.4710, test_macro_f1=0.3824, seconds=0.363


refit epoch 04: loss=1.35464, accuracy=0.4213, macro_f1=0.3579, test_accuracy=0.4509, test_macro_f1=0.3751, seconds=0.373


refit epoch 05: loss=1.32889, accuracy=0.4399, macro_f1=0.3794, test_accuracy=0.4332, test_macro_f1=0.3477, seconds=0.384


... (13 outputs left out)

**Confusion matrix** — rows: actual; columns: predicted.

,Negative,Positive,Irrelevant,Neutral
Negative,64,26,22,8
Positive,19,62,16,9
Irrelevant,19,20,16,11
Neutral,22,25,17,41


**One-vs-rest counts for each class**

,label_id,label,TP,FP,FN,TN,support
0,0,Negative,64,60,56,217,120
1,1,Positive,62,71,44,220,106
2,2,Irrelevant,16,55,50,276,66
3,3,Neutral,41,28,64,264,105


## Optional continuation
Set `EXTRA_EPOCHS` in the configuration cell before running the notebook. This cell explicitly reloads the completed refit checkpoint, including AdamW and RNG state, and adds epochs to its saved count. It does not restart training. A fresh run directory and the selected-duration checkpoint preserve the shorter experiment.

In [15]:
if EXTRA_EPOCHS:
    checkpoint = torch.load(RUN_DIR / "model.pt", map_location="cpu", weights_only=True)
    assert checkpoint["pool_fingerprint"] == pool_fingerprint(pool)
    assert checkpoint["vocab"] == vocab and checkpoint["config"] == CONFIG
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    restore_rng_state(checkpoint["rng_state"])
    start_epoch = checkpoint["completed_epochs"]
    refit_history = list(checkpoint["refit_history"])
    for epoch in range(start_epoch + 1, start_epoch + EXTRA_EPOCHS + 1):
        started = time.perf_counter()
        model.embedding.weight.requires_grad_(epoch > CONFIG["freeze_epochs"])
        metrics = train_epoch(model, pool_loader, optimizer, loss_fn, device)
        metrics.update(test_snapshot(model, test_loader, device, loss_fn))
        refit_history.append(logger.record("continuation", epoch, metrics, model, optimizer, started))
        save_refit_checkpoint(RUN_DIR / "model.pt", model, optimizer, vocab, CONFIG,
                              epoch, refit_history, pool)
    final_test_report = export_evaluation("test_final", test, model, vocab)
else:
    final_test_report = selected_test_report
    print("No additional epochs requested; final model is the selected-duration refit.")
pd.DataFrame(refit_history).to_csv(RUN_DIR / "refit_history.csv", index=False)
result = dict(config=CONFIG, selected_epoch=selected_epoch, completed_epochs=refit_history[-1]["epoch"],
              validation_macro_f1=best_validation_f1, validation=validation_report,
              selected_duration_test=selected_test_report, final_test=final_test_report,
              parameters=parameters, split=dict(train=473, validation=118, pool=591, test=397),
              selection_history=selection_history, refit_history=refit_history)
(RUN_DIR / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

No additional epochs requested; final model is the selected-duration refit.


21511

### Final test evaluation — run this cell to compute all metrics
This cell performs inference on all 397 test rows using the current trained model. It calculates and displays accuracy; macro and weighted precision, recall and F1; per-class scores and support; the confusion matrix image and table; and per-class TP, FP, FN and TN. All values come from predictions generated by this execution. Results and prediction IDs are also saved in the current run directory.

In [16]:
# Recompute from the trained model; no previously saved metrics are needed.
final_test_report = export_evaluation("test_final", test, model, vocab)
result["final_test"] = final_test_report
(RUN_DIR / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print("Saved all computed metrics, predictions and the confusion matrix in:", RUN_DIR.name)

### test_final: evaluation on 397 examples

**Overall metrics** — P, R and F1 are shown for both macro and weighted averaging.

,average,accuracy,precision,recall,f1,n
0,macro,0.461,0.4505,0.4378,0.4371,397
1,weighted,0.461,0.4751,0.4610,0.4606,397


**Per-class precision, recall, F1 and support**

,precision,recall,f1-score,support
Negative,0.5161,0.5333,0.5246,120
Positive,0.4662,0.5849,0.5188,106
Irrelevant,0.2254,0.2424,0.2336,66
Neutral,0.5942,0.3905,0.4713,105


**Confusion matrix** — rows: actual; columns: predicted.

,Negative,Positive,Irrelevant,Neutral
Negative,64,26,22,8
Positive,19,62,16,9
Irrelevant,19,20,16,11
Neutral,22,25,17,41


**One-vs-rest counts for each class**

,label_id,label,TP,FP,FN,TN,support
0,0,Negative,64,60,56,217,120
1,1,Positive,62,71,44,220,106
2,2,Irrelevant,16,55,50,276,66
3,3,Neutral,41,28,64,264,105


Saved all computed metrics, predictions and the confusion matrix in: 20261121T165235_181853Z


## Learning curves and comparison with the shared baseline
The two plots keep validation-based selection separate from full-pool refitting. Curves show unsmoothed observations. Training and test accuracy may differ partly because dropout is enabled only during training. The baseline row is a recorded group result on the same 397-row file, not a baseline retrained in this BiLSTM notebook.

In [17]:
def plot_history(history, second, filename, title):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, key in zip(axes, ["loss", "accuracy", "macro_f1"]):
        for phase, color in [("train", "#1e40af"), (second, "#60a5fa")]:
            ax.plot(frame.epoch, frame[f"{phase}_{key}"], label=phase, color=color)
        ax.set(xlabel="Epoch", title=key.replace("_", " "))
        if key != "loss": ax.set_ylim(0, 1)
        ax.grid(alpha=.2)
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(RUN_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_history(selection_history, "validation", "selection_curves.png", "Selection: 473 train / 118 validation")
plot_history(refit_history, "test", "train_test_curves.png", "Refit: 591 train / 397 previously inspected test")
# Recorded reports/baseline_metrics.json summary; not a new baseline fit.
baseline = {'accuracy': 0.5063, 'macro_precision': 0.533, 'macro_recall': 0.4693, 'macro_f1': 0.4662}
def metric_row(name, report):
    return dict(model=name, accuracy=report["accuracy"], P=report["macro avg"]["precision"],
                R=report["macro avg"]["recall"], F1=report["macro avg"]["f1-score"])
comparison = pd.DataFrame([
    dict(model="Recorded TF-IDF + LR baseline", accuracy=baseline["accuracy"],
         P=baseline["macro_precision"], R=baseline["macro_recall"], F1=baseline["macro_f1"]),
    metric_row(f"{CONFIG['embedding']} + BiLSTM (selected duration)", selected_test_report),
] + ([metric_row("BiLSTM after continuation", final_test_report)] if EXTRA_EPOCHS else []))
comparison.to_csv(RUN_DIR / "baseline_comparison.csv", index=False)
display(comparison.round(4))

,model,accuracy,P,R,F1
0,Recorded TF-IDF + LR baseline,0.5063,0.5330,0.4693,0.4662
1,glove-twitter-50 + BiLSTM (selected duration),0.4610,0.4505,0.4378,0.4371


## Recorded iterations and tuning evidence
The following compact tables transcribe saved historical reports; they are not newly executed trials. Each row records a source run. The 25-dimensional validation search tried hidden sizes {16, 48} × frozen/fine-tuned embeddings, with dropout 0.4, seed 42, a 40-epoch cap and patience 8 (up to 160 selection epochs total). Fine-tuned trials used three initial frozen epochs.

The separate 50/100-dimensional Twitter experiments fine-tuned from epoch 1. Corpus-trained GloVe experiments fitted word co-occurrences rather than using external vectors. The original 25-dimensional 20/40/60-epoch chain used a vocabulary built from the whole 591-row pool even during selection; this includes validation-text vocabulary exposure. These experiments differ in more than embedding dimension and therefore are **not a controlled dimension-only ablation**. The main implementation above uses the fitting partition only.

Only one seed was used for these runs. Do not interpret small score differences as statistically established improvements, or present additional dimensions/seeds as distinct classifier families. Historical test-driven observations are disclosed rather than described as an untouched evaluation.

In [18]:
# Historical summaries from saved runs; trial source: runs/glove-comparison/20261121T095902_562974Z/tuning.csv
historical = pd.DataFrame([{'experiment': 'Twitter25_e20', 'epochs': 20, 'accuracy': 0.4106, 'macro_f1': 0.4073, 'source': 'runs/glove-bilstm/20261119T104915_035482Z'}, {'experiment': 'Twitter25_e40', 'epochs': 40, 'accuracy': 0.4433, 'macro_f1': 0.4348, 'source': 'runs/glove-bilstm/20261119T105807_895659Z'}, {'experiment': 'Twitter25_e60', 'epochs': 60, 'accuracy': 0.3753, 'macro_f1': 0.3771, 'source': 'runs/glove-bilstm/20261119T111035_354122Z'}, {'experiment': 'Twitter50_e12', 'epochs': 12, 'accuracy': 0.4609571788413098, 'macro_f1': 0.4370649064922824, 'source': 'runs/glove-twitter-bilstm/20261121T125121_126917Z'}, {'experiment': 'Twitter50_e32', 'epochs': 32, 'accuracy': 0.4055, 'macro_f1': 0.3951, 'source': 'runs/glove-twitter-bilstm/20261121T144026_328332Z'}, {'experiment': 'Twitter100_e14', 'epochs': 14, 'accuracy': 0.4634760705289673, 'macro_f1': 0.4326954633244443, 'source': 'runs/glove-twitter-bilstm/20261121T125224_929082Z'}, {'experiment': 'Corpus200_e20', 'epochs': 20, 'accuracy': 0.3979848866498741, 'macro_f1': 0.3923503507853078, 'source': 'runs/glove-corpus-bilstm/20261121T121122_115631Z'}, {'experiment': 'Corpus100_e12', 'epochs': 12, 'accuracy': 0.3879093198992443, 'macro_f1': 0.3684624612611631, 'source': 'runs/glove-corpus-bilstm/20261121T121522_989774Z'}, {'experiment': 'Corpus100_e32', 'epochs': 32, 'accuracy': 0.4181, 'macro_f1': 0.3997, 'source': 'runs/glove-corpus-bilstm/20261121T122754_238205Z'}])
trials = pd.DataFrame([{'model': 'BiLSTM', 'hidden': 16, 'dropout': 0.4, 'finetune': False, 'best_epoch': 12, 'validation_macro_f1': 0.4134071810542399}, {'model': 'BiLSTM', 'hidden': 16, 'dropout': 0.4, 'finetune': True, 'best_epoch': 12, 'validation_macro_f1': 0.4037367292631397}, {'model': 'BiLSTM', 'hidden': 48, 'dropout': 0.4, 'finetune': False, 'best_epoch': 12, 'validation_macro_f1': 0.468524531024531}, {'model': 'BiLSTM', 'hidden': 48, 'dropout': 0.4, 'finetune': True, 'best_epoch': 12, 'validation_macro_f1': 0.4528769513548311}])
display(trials.round(4))
display(historical.round(4))
fig, ax = plt.subplots(figsize=(8, 4))
for prefix, color in [("Twitter25", "#1e40af"), ("Twitter50", "#60a5fa")]:
    chain = historical[historical.experiment.str.startswith(prefix)].sort_values("epochs")
    ax.plot(chain.epochs, chain.macro_f1, marker="o", color=color, label=prefix)
ax.set(xlabel="Cumulative refit epoch", ylabel="Test macro-F1", title="Recorded continuation experiments")
ax.legend(); ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_DIR / "historical_continuation.png", dpi=180)
plt.show(); plt.close(fig)
print("25d: F1 rose from 0.4073 at epoch 20 to 0.4348 at 40, then fell to 0.3771 at 60.")
print("50d: F1 fell from 0.4371 at epoch 12 to 0.3951 at 32.")
print("Longer training did not reliably improve generalization; historical results remained below baseline F1 0.4662.")

,model,hidden,dropout,finetune,best_epoch,validation_macro_f1
0,BiLSTM,16,0.4,False,12,0.4134
1,BiLSTM,16,0.4,True,12,0.4037
2,BiLSTM,48,0.4,False,12,0.4685
3,BiLSTM,48,0.4,True,12,0.4529


,experiment,epochs,accuracy,macro_f1,source
0,Twitter25_e20,20,0.4106,0.4073,runs/glove-bilstm/20261119T104915_035482Z
1,Twitter25_e40,40,0.4433,0.4348,runs/glove-bilstm/20261119T105807_895659Z
2,Twitter25_e60,60,0.3753,0.3771,runs/glove-bilstm/20261119T111035_354122Z
3,Twitter50_e12,12,0.4610,0.4371,runs/glove-twitter-bilstm/20261121T125121_126917Z
4,Twitter50_e32,32,0.4055,0.3951,runs/glove-twitter-bilstm/20261121T144026_328332Z
5,Twitter100_e14,14,0.4635,0.4327,runs/glove-twitter-bilstm/20261121T125224_929082Z
6,Corpus200_e20,20,0.3980,0.3924,runs/glove-corpus-bilstm/20261121T121122_115631Z
7,Corpus100_e12,12,0.3879,0.3685,runs/glove-corpus-bilstm/20261121T121522_989774Z
8,Corpus100_e32,32,0.4181,0.3997,runs/glove-corpus-bilstm/20261121T122754_238205Z


25d: F1 rose from 0.4073 at epoch 20 to 0.4348 at 40, then fell to 0.3771 at 60.
50d: F1 fell from 0.4371 at epoch 12 to 0.3951 at 32.
Longer training did not reliably improve generalization; historical results remained below baseline F1 0.4662.


## Error analysis and reload verification
Show actual misclassified examples with stable row IDs and actual and predicted labels. The most frequent confusion and weakest class are computed from predictions, not assumed in advance. The final model is reloaded from disk and must reproduce every saved final prediction. These examples support discussion but are not used for training or automatic configuration selection.

In [19]:
prefix = "test_final"
predictions = pd.read_csv(RUN_DIR / f"{prefix}_predictions.csv")
errors = predictions[predictions.Sentiment != predictions.prediction].copy()
errors["actual_label"] = errors.Sentiment.map(dict(enumerate(LABEL_NAMES)))
errors["predicted_label"] = errors.prediction.map(dict(enumerate(LABEL_NAMES)))
errors.to_csv(RUN_DIR / "errors.csv", index=False)
display(errors[["row_id", "actual_label", "predicted_label"]].head(6))
cm = confusion_matrix(predictions.Sentiment, predictions.prediction, labels=LABEL_IDS)
off_diag = cm.copy(); np.fill_diagonal(off_diag, 0)
actual, predicted = np.unravel_index(off_diag.argmax(), off_diag.shape)
weakest = min(LABEL_NAMES, key=lambda label: final_test_report[label]["f1-score"])
print(f"Most common off-diagonal error: {LABEL_NAMES[actual]} -> {LABEL_NAMES[predicted]} ({off_diag[actual,predicted]} rows).")
print(f"Lowest per-class F1: {weakest}, {final_test_report[weakest]['f1-score']:.4f}.")
checkpoint = torch.load(RUN_DIR / "model.pt", map_location="cpu", weights_only=True)
assert checkpoint["pool_fingerprint"] == pool_fingerprint(pool)
reloaded = BiLSTMClassifier(checkpoint["model_state_dict"]["embedding.weight"].numpy(),
                            CONFIG["hidden_size"], CONFIG["dropout"]).to(device)
reloaded.load_state_dict(checkpoint["model_state_dict"])
y, p = evaluate(reloaded, make_loader(test, checkpoint["vocab"], CONFIG["batch_size"], False), device)
np.testing.assert_array_equal(predictions.row_id, test.row_id)
np.testing.assert_array_equal(predictions.prediction, p)
assert checkpoint["completed_epochs"] == selected_epoch + EXTRA_EPOCHS
print(f"Verified: all {len(p)} predictions match after checkpoint reload.")

,row_id,actual_label,predicted_label
1,test_1,Negative,Irrelevant
2,test_2,Negative,Positive
3,test_3,Negative,Irrelevant
6,test_6,Negative,Neutral
11,test_11,Negative,Irrelevant
13,test_13,Negative,Irrelevant


Most common off-diagonal error: Negative -> Positive (26 rows).
Lowest per-class F1: Irrelevant, 0.2336.
Verified: all 397 predictions match after checkpoint reload.


## Predict from shared-cleaned input text
This interface expects text processed by the group's cleaning pipeline. Passing arbitrary raw text would skip that shared preprocessing stage. The probabilities below use softmax; the training loss above directly uses logits.

In [20]:
def predict_cleaned_text(texts):
    frame = pd.DataFrame({"text_clean": list(texts), "Sentiment": 0})
    rows = []
    reloaded.eval()
    with torch.no_grad():
        for ids, lengths, _ in make_loader(frame, vocab, CONFIG["batch_size"], False):
            rows.extend(torch.softmax(reloaded(ids.to(device), lengths), dim=1).cpu().numpy())
    probabilities = pd.DataFrame(rows, columns=LABEL_NAMES)
    probabilities.insert(0, "input_text", list(texts))
    probabilities["predicted_label"] = probabilities[LABEL_NAMES].idxmax(axis=1)
    return probabilities

sample = test.head(3)
probs = predict_cleaned_text(sample.text_clean.tolist()).drop(columns="input_text")
probs.insert(0, "row_id", sample.row_id.tolist())
display(probs.round(4))

,row_id,Negative,Positive,Irrelevant,Neutral,predicted_label
0,test_0,0.3337,0.1753,0.2737,0.2173,Negative
1,test_1,0.1441,0.1854,0.4273,0.2432,Irrelevant
2,test_2,0.1596,0.4423,0.2660,0.1321,Positive


## Contribution, interpretation and handoff
**Yuchen Gu's contribution:** GloVe-based sequence-classification implementation; experiments with externally pretrained and corpus-trained embeddings; training and continuation; validation tuning; per-epoch recording; metric exports, confusion matrices, and error analysis. The shared data cleaning and TF-IDF baseline belong to the group pipeline.

The BiLSTM adds sequence context to word representations. However, the recorded results do not demonstrate superiority to the TF-IDF baseline. The small fitting pool, class imbalance, and increased parameter count are plausible contributors, not proven causes. The continuation evidence shows that lower training loss need not translate to higher test accuracy or F1. This classifier uses text only and does not explicitly condition on a target entity, which may limit its ability to distinguish entity-specific sentiment and irrelevance.

**Handoff:** merge this notebook's source and explanations into the group notebook; keep the shared cleaning stage ahead of it; use matching test IDs in shared comparison tables. The notebook preserves metric tables; running it generates figures, while saved image outputs are cleared for readability. Each output directory includes complete checkpoints, configuration, IDs, predictions, per-class reports, confusion matrices/counts, curves, and epoch logs. Historical summaries identify their source runs. Data are read from the local processed directory; pretrained vectors use the Gensim cache/download mechanism. Hardware/library differences can change a rerun; use the recorded artifacts to identify which experiment a reported value comes from.

**References**
- Pennington, Socher & Manning (2014), [GloVe: Global Vectors for Word Representation](https://aclanthology.org/D14-1162/).
- [Stanford GloVe project and Twitter embeddings](https://nlp.stanford.edu/projects/glove/).
- [Gensim pretrained model catalog](https://radimrehurek.com/gensim/downloader.html).